# AmbiGAN Boundary Generation (224x224)

Generator and discriminator operate directly at 224x224. Core logic lives in `src/ambigan/`; this notebook only validates input and launches the YAML-driven pipeline.

In [1]:
from pathlib import Path
import sys

project_root = Path.cwd()
if not (project_root / 'configs').exists():
    project_root = project_root.parent

full_config = project_root / 'configs/experiments/ambigan_boundary_generation.yaml'
smoke_config = project_root / 'configs/experiments/ambigan_boundary_generation_smoke.yaml'
lung_app_python = Path.home() / '.conda/envs/lung_app/python.exe'
python_executable = lung_app_python if lung_app_python.exists() else Path(sys.executable)
data_dir = project_root / 'data/raw/chest_xray_2018'
image_count = sum(1 for path in data_dir.rglob('*') if path.is_file())

print(f'Notebook kernel : {sys.executable}')
print(f'Training Python : {python_executable}')
config = smoke_config if image_count < 20 else full_config
mode = 'SMOKE TEST' if config == smoke_config else 'FULL TRAINING'

print(f'Mode            : {mode}')
print(f'Config          : {config}')
print(f'Dataset         : {data_dir}')
print(f'Image count     : {image_count}')

assert config.exists(), f'Missing config: {config}'
assert python_executable.exists(), f'Missing Python interpreter: {python_executable}'
assert image_count > 0, (
    'Dataset is empty. Add images under train/NORMAL, train/PNEUMONIA, '
    'test/NORMAL, and test/PNEUMONIA, or change dataset.data_dir in YAML.'
)

Notebook kernel : C:\Users\lenovo\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\python.exe
Training Python : C:\Users\lenovo\.conda\envs\lung_app\python.exe
Mode            : SMOKE TEST
Config          : d:\NAM_3\DEEPLEARNING\xray_pneumonia_project\configs\experiments\ambigan_boundary_generation_smoke.yaml
Dataset         : d:\NAM_3\DEEPLEARNING\xray_pneumonia_project\data\raw\chest_xray_2018
Image count     : 8


In [2]:
import subprocess

command = [
    str(python_executable),
    str(project_root / 'scripts/run_ambigan_boundary_generation.py'),
    '--config', str(config),
]
result = subprocess.run(
    command,
    cwd=project_root,
    text=True,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
)
print(result.stdout)
if result.returncode != 0:
    raise RuntimeError(f'AmbiGAN process failed with exit code {result.returncode}')

Epoch 001/1 | train_loss=0.6925 | val_loss=0.6920 | val_acc=0.5000 | val_f1=0.0000
2026-06-13 15:50:01 | INFO | Epoch 001/1 | train_loss=0.6925 | val_loss=0.6920 | val_acc=0.5000 | val_f1=0.0000
DCGAN 001/1 | G=0.6940 D=1.4089
AmbiGAN 001/1 | G=0.4619 GAN=0.6922 Amb=-1.1513 D=1.3862
Boundary images: 2
Run directory: D:\NAM_3\DEEPLEARNING\xray_pneumonia_project\outputs\experiments\ambigan_boundary_generation_smoke\20260613_155001_351367

